# Antardrishti — does a terrain check remove false flood water?

**Question.** Radar calls dark pixels water. Hill slopes in radar shadow and smooth dry ground far above any river are dark too, and a flood cannot sit there. Operational services mask such pixels with **HAND** (height above nearest drainage) — TU Wien's algorithm in the Copernicus Global Flood Monitoring service does, and published tests found about 15 m suitable. Does it help **our** rule, on **our** test split, and does it delete real Indian flood water?

**Rule scored.** `water = dark AND NOT (HAND > h) AND NOT (slope > s)`, on top of the shipped darkness rule at each scale. Missing terrain keeps the darkness verdict.

**Protocol.** HAND source (MERIT Hydro 90 m, or GlobalHAND 30 m if fetched), *h* and *s* are chosen on the **train split only, at 200 m**. Scored on test, valid and Bolivia at 10, 100 and 200 m. India: valid + test chips only.

**Decision rule — fixed before running, not after reading the numbers:**
1. Test-split IoU at 200 m must rise, **and** the paired bootstrap 95% range of the gain must lie above zero.
2. Recall on held-out Indian chips must not fall by more than **0.02**.

Fail either and the check does not ship; the result is reported as a measured negative, like region growing in notebook 07.

**Before running:** `python -m scripts.fetch_terrain --sen1floods11 C:\sen1floods11` (from `backend/`). Chips without terrain are left out of BOTH rules' scores, and how many is printed.

In [1]:
!pip install rasterio numpy scipy


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
LOCAL = r"C:\sen1floods11"
TERRAIN = None            # default: <LOCAL>/Terrain, as fetch_terrain.py writes it

import sys, json, math, time, importlib
from pathlib import Path

import numpy as np
import rasterio

root = Path(LOCAL)
terrain_dir = Path(TERRAIN) if TERRAIN else root / "Terrain"
EVALUATION = Path.cwd().resolve().parent.parent / "evaluation"
sys.path.insert(0, str(EVALUATION))

import spatial as SP, change as C, events as EV, terrain as T
for module in (SP, C, EV, T):
    importlib.reload(module)

spatial = json.loads((EVALUATION / "spatial_results.json").read_text(encoding="utf-8"))
DB = {10: -20.0,
      10 * 10: spatial["step1_best_threshold_at_scale"]["100 m"]["db"],
      10 * 20: spatial["step1_best_threshold_at_scale"]["200 m"]["db"]}
print("darkness thresholds by scale (notebooks 04 and 07):", DB)

s1 = {p.name.replace("_S1Hand.tif", ""): p for p in root.rglob("*_S1Hand.tif")}
labels = {p.name.replace("_LabelHand.tif", ""): p for p in root.rglob("*_LabelHand.tif")}
terrain = {p.name.replace("_Terrain.tif", ""): p for p in terrain_dir.glob("*_Terrain.tif")}
splits = SP.load_splits(root, hand_keys=set(labels))
split_of = {k: name for name, keys in splits.items() for k in keys}
keys = sorted(k for k in split_of if k in s1 and k in labels)
missing = [k for k in keys if k not in terrain]
print(f"{len(keys)} labelled chips in the splits; {len(keys) - len(missing)} have terrain, "
      f"{len(missing)} do not (left out of every score below)")

darkness thresholds by scale (notebooks 04 and 07): {10: -20.0, 100: -19.0, 200: -18.5}
446 labelled chips in the splits; 446 have terrain, 0 do not (left out of every score below)


In [3]:
def load(key):
    with rasterio.open(s1[key]) as src:
        vv = src.read(1).astype(np.float32); vh = src.read(2).astype(np.float32)
    vv[vv < C.NODATA_FLOOR_DB] = np.nan; vh[vh < C.NODATA_FLOOR_DB] = np.nan
    with rasterio.open(labels[key]) as src:
        truth = src.read(1).astype(np.int8)
    with rasterio.open(terrain[key]) as src:
        names = list(src.descriptions)
        bands = {n: src.read(i + 1).astype(np.float32) for i, n in enumerate(names)}
    truth = truth.copy(); truth[~(np.isfinite(vv) & np.isfinite(vh))] = -1
    return C.fused(vv, vh), truth, bands

# Coarse scales are kept in memory; 10 m is re-read later for the chosen rule
# only (446 full-resolution chips with terrain would not fit comfortably).
chips, started = [], time.time()
SOURCES = set()
for i, key in enumerate(k for k in keys if k in terrain):
    fused, truth, bands = load(key)
    if not (truth != -1).any():
        continue
    chip = {"key": key, "split": split_of[key]}
    for factor in (10, 20):
        f = SP.block_mean(fused, factor)
        chip[f"truth{factor}"] = SP.block_label(truth, factor)
        chip[f"dark{factor}"] = np.isfinite(f) & (f < DB[10 * factor])
        chip[f"slope{factor}"] = SP.block_mean(bands["slope_deg"], factor)
        for source in ("hand_merit", "hand_30m"):
            if source in bands and np.isfinite(bands[source]).any():
                chip[f"{source}{factor}"] = SP.block_mean(bands[source], factor)
                SOURCES.add(source)
    chips.append(chip)
    if (i + 1) % 50 == 0:
        print(f"  {i + 1} chips, {time.time() - started:.0f}s")
by_split = {s: [c for c in chips if c["split"] == s] for s in ("train", "valid", "test", "bolivia")}
print({s: len(v) for s, v in by_split.items()}, "HAND sources present:", sorted(SOURCES))

  50 chips, 4s
  100 chips, 8s
  150 chips, 12s
  200 chips, 15s
  250 chips, 19s
  300 chips, 23s
  350 chips, 27s
  400 chips, 31s
{'train': 251, 'valid': 86, 'test': 88, 'bolivia': 15} HAND sources present: ['hand_30m', 'hand_merit']


## Choose on train, at 200 m

In [4]:
best = None
# MERIT first: it is in the Earth Engine catalogue, so on a tie it wins - a
# community asset has to do strictly better to be chosen.
for source in [s for s in ("hand_merit", "hand_30m") if s in SOURCES]:
    usable = [c for c in by_split["train"] if f"{source}20" in c]
    table = T.sweep(usable, "dark20", hand_key=f"{source}20", slope_key="slope20", truth_key="truth20")
    off = next(r for r in table if math.isinf(r["max_hand_m"]) and math.isinf(r["max_slope_deg"]))
    print(f"\n{source}: train chips {len(usable)}; no check: IoU {off['iou']:.3f}")
    print(f"{'HAND <=':>9}{'slope <=':>10}{'IoU':>8}{'prec':>8}{'recall':>8}")
    for r in table[:8]:
        print(f"{r['max_hand_m']:>9}{r['max_slope_deg']:>10}{r['iou']:>8.3f}{r['precision']:>8.3f}{r['recall']:>8.3f}")
    if best is None or table[0]["iou"] > best[1]["iou"]:
        best = (source, table[0])

SOURCE, CHOICE = best
print(f"\nchosen on train: {SOURCE}, HAND <= {CHOICE['max_hand_m']} m, slope <= {CHOICE['max_slope_deg']} deg")
if math.isinf(CHOICE["max_hand_m"]) and math.isinf(CHOICE["max_slope_deg"]):
    print("The best train rule is NO check at all: terrain does not help on train.")


hand_merit: train chips 251; no check: IoU 0.567
  HAND <=  slope <=     IoU    prec  recall
     10.0       inf   0.584   0.790   0.691
     10.0      30.0   0.584   0.790   0.691
     10.0      20.0   0.584   0.790   0.691
     10.0      15.0   0.584   0.790   0.691
     10.0      10.0   0.584   0.790   0.691
     10.0       5.0   0.580   0.792   0.685
     15.0       inf   0.579   0.778   0.693
     15.0      30.0   0.579   0.778   0.693

hand_30m: train chips 251; no check: IoU 0.567
  HAND <=  slope <=     IoU    prec  recall
     10.0      10.0   0.578   0.800   0.676
     10.0       inf   0.578   0.799   0.676
     10.0      30.0   0.578   0.799   0.676
     10.0      20.0   0.578   0.799   0.676
     10.0      15.0   0.578   0.799   0.676
     15.0       inf   0.577   0.785   0.686
     15.0      30.0   0.577   0.785   0.686
     15.0      20.0   0.577   0.785   0.686

chosen on train: hand_merit, HAND <= 10.0 m, slope <= inf deg


## Score where the choice was never made: test, valid, Bolivia

In [5]:
H, S = CHOICE["max_hand_m"], CHOICE["max_slope_deg"]
results, gains = {}, {}
print(f"{'split':<9}{'scale':>6}{'IoU before':>12}{'IoU after':>11}{'prec':>14}{'recall':>15}")
for split in ("test", "valid", "bolivia"):
    for factor in (10, 20):
        usable = [c for c in by_split[split] if f"{SOURCE}{factor}" in c]
        keys_ = dict(hand_key=f"{SOURCE}{factor}", slope_key=f"slope{factor}", truth_key=f"truth{factor}")
        base_c = T.chip_counts(usable, f"dark{factor}", **keys_)
        new_c = T.chip_counts(usable, f"dark{factor}", H, S, **keys_)
        b, n = T.pooled(base_c), T.pooled(new_c)
        label = f"{factor * 10} m"
        results[f"{split} {label}"] = {"before": b, "after": n,
                                       "removed": T.removed_breakdown(usable, f"dark{factor}", H, S, **keys_)}
        gains[f"{split} {label}"] = T.paired_bootstrap_gain(base_c, new_c)
        print(f"{split:<9}{label:>6}{b['iou']:>12.3f}{n['iou']:>11.3f}"
              f"{b['precision']:>7.3f}->{n['precision']:.3f}{b['recall']:>8.3f}->{n['recall']:.3f}")
print("\nWhat was removed on test at 200 m:", results["test 200 m"]["removed"])

split     scale  IoU before  IoU after          prec         recall
test      100 m       0.570      0.579  0.808->0.843   0.659->0.650
test      200 m       0.609      0.621  0.807->0.841   0.713->0.703
valid     100 m       0.564      0.568  0.807->0.818   0.653->0.650
valid     200 m       0.594      0.597  0.801->0.809   0.696->0.695
bolivia   100 m       0.682      0.682  0.919->0.919   0.725->0.725
bolivia   200 m       0.718      0.718  0.913->0.913   0.770->0.770

What was removed on test at 200 m: {'removed_dry_px': 213, 'removed_water_px': 54, 'share_dry': 0.797752808988764}


In [6]:
# 10 m, re-read for the chosen rule only.
counts10 = {"base": [], "new": []}
for c in by_split["test"]:
    fused, truth, bands = load(c["key"])
    dark = np.isfinite(fused) & (fused < DB[10])
    hand = bands.get(SOURCE)
    if hand is None:
        continue
    counts10["base"].append(T._counts(dark, truth))
    counts10["new"].append(T._counts(T.apply(dark, hand, bands["slope_deg"], H, S), truth))
b10, n10 = T.pooled(counts10["base"]), T.pooled(counts10["new"])
results["test 10 m"] = {"before": b10, "after": n10}
gains["test 10 m"] = T.paired_bootstrap_gain(counts10["base"], counts10["new"])
print(f"test 10 m: IoU {b10['iou']:.3f} -> {n10['iou']:.3f}, precision {b10['precision']:.3f} -> "
      f"{n10['precision']:.3f}, recall {b10['recall']:.3f} -> {n10['recall']:.3f}")

test 10 m: IoU 0.515 -> 0.521, precision 0.795 -> 0.830, recall 0.594 -> 0.583


## Per event at 200 m — where does it help, where does it hurt?

All chips per event (tuned and untuned mixed): read for the spread between landscapes. Somalia and Pakistan are the dry-ground false-water cases; Indian hill chips, if any, are the radar-shadow case.

In [7]:
usable = [c for c in chips if f"{SOURCE}20" in c]
keys20 = dict(hand_key=f"{SOURCE}20", slope_key="slope20", truth_key="truth20")
def before(c): return T.chip_counts([c], "dark20", **keys20)[0]
def after(c): return T.chip_counts([c], "dark20", H, S, **keys20)[0]
ev_before, ev_after = EV.by_event(usable, before), EV.by_event(usable, after)
print(f"{'event':<12}{'IoU':>14}{'precision':>16}{'recall':>16}")
for event in sorted(ev_before, key=lambda e: ev_after[e]["iou"] - ev_before[e]["iou"]):
    b, a = ev_before[event], ev_after[event]
    print(f"{event:<12}{b['iou']:>6.3f}->{a['iou']:.3f}{b['precision']:>8.3f}->{a['precision']:.3f}"
          f"{b['recall']:>8.3f}->{a['recall']:.3f}")

event                  IoU       precision          recall
India        0.628->0.611   0.880->0.883   0.687->0.665
Mekong       0.835->0.833   0.990->0.990   0.842->0.840
Nigeria      0.669->0.669   0.973->0.974   0.682->0.681
Bolivia      0.718->0.718   0.913->0.913   0.770->0.770
Ghana        0.452->0.452   0.813->0.813   0.504->0.504
Somalia      0.163->0.163   0.217->0.217   0.393->0.393
Sri-Lanka    0.672->0.675   0.952->0.958   0.696->0.696
Spain        0.776->0.786   0.880->0.898   0.868->0.864
Pakistan     0.240->0.251   0.324->0.345   0.480->0.480
USA          0.526->0.574   0.590->0.654   0.828->0.824
Paraguay     0.473->0.527   0.728->0.867   0.574->0.574


## India, held out

In [8]:
india_before = EV.india_summary(usable, before)["held_out"]
india_after = EV.india_summary(usable, after)["held_out"]
print(f"India valid + test, 200 m: IoU {india_before['iou']:.3f} -> {india_after['iou']:.3f}  "
      f"recall {india_before['recall']:.3f} -> {india_after['recall']:.3f}  "
      f"({india_after['chips']} chips, 95% range after {india_after['ci95'][0]:.3f}-{india_after['ci95'][1]:.3f})")

India valid + test, 200 m: IoU 0.725 -> 0.705  recall 0.787 -> 0.764  (28 chips, 95% range after 0.555-0.804)


## The decision

In [9]:
test200 = results["test 200 m"]
decision = T.decide(test200["before"], test200["after"], gains["test 200 m"],
                    india_before if india_before["chips"] else None,
                    india_after if india_after["chips"] else None)
print("SHIP" if decision["ship"] else "DO NOT SHIP")
for reason in decision["reasons"]:
    print("  -", reason)

india_block = {"event": "Assam, August 2016", "chips": india_after["chips"],
               "iou": round(india_after["iou"], 3), "precision": round(india_after["precision"], 3),
               "recall": round(india_after["recall"], 3),
               "ci95": [round(v, 3) for v in india_after["ci95"]]}
block = T.backend_block(CHOICE, test200["after"], india_block,
                        {"hand_merit": "MERIT/Hydro/v1_0_1", "hand_30m": "users/gena/GlobalHAND/30m/hand-1000"}[SOURCE],
                        decision)
finite = lambda v: None if isinstance(v, float) and math.isinf(v) else v
payload = {
    "protocol": "thresholds chosen on train at 200 m; scored on test/valid/bolivia; decision rule fixed before the run",
    "decision_rule": {"test_iou_must_rise": True, "gain_ci_must_exclude_zero": True,
                      "india_recall_tolerance": T.INDIA_RECALL_TOLERANCE},
    "chips_with_terrain": len(chips), "chips_without_terrain": len(missing),
    "source": SOURCE,
    "choice": {"max_hand_m": finite(H), "max_slope_deg": finite(S)},
    "results": results,
    "gain_ci95": {k: list(v) for k, v in gains.items()},
    "per_event_200m": {"before": ev_before, "after": ev_after},
    "india_200m": {"before": india_before, "after": india_after},
    "decision": decision,
    "terrain_rule_for_backend": block,
}
out = EVALUATION / "terrain_results.json"
out.write_text(json.dumps(T.plain(payload), indent=2, allow_nan=False), encoding="utf-8")
print(f"\nwritten to {out}")
if block:
    print("\nSet TERRAIN_RULE in backend/detection/sar.py to:\n")
    print(json.dumps(T.plain(block), indent=4))
else:
    print("\nTERRAIN_RULE stays None. Record this as a measured negative in the report.")

DO NOT SHIP
  - test IoU rose 0.609 -> 0.621 (+0.012)
  - the gain's 95% range (-0.010 to 0.051) does not exclude zero
  - Indian recall fell by 0.024, more than the 0.02 allowed

written to C:\Users\punit\OneDrive\Desktop\Group_project\application\antardrishti\evaluation\terrain_results.json

TERRAIN_RULE stays None. Record this as a measured negative in the report.


## What to do with the result

- **SHIP:** paste the printed block into `TERRAIN_RULE` in `backend/detection/sar.py`. The detection rule's version changes with it, so cached results are recomputed rather than served. Every radar flood result then reports `water_excluded_by_terrain`, and users can switch the check off. A test ties `TERRAIN_RULE` to `terrain_results.json`.
- **DO NOT SHIP:** leave `TERRAIN_RULE = None`. The per-event table says where terrain helped and where it hurt — that is still a finding for the report.
- Either way, read the *removed* breakdown: a check that removes as much true water as dry ground is trading recall for precision, whatever the IoU says.